In [8]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
import random
import hashlib
import os


# REPRODUCTIBILITÉ

In [9]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)


# 1.  RÉSEAU DE 10 PHARMACIES NATIONALES (Maroc — exemple réaliste)

In [10]:
PHARMACIES = [
    {"Pharmacy_ID": "PH001", "Pharmacy_Name": "Pharmacie Al Amal",     "City": "Casablanca", "Region": "Grand Casablanca-Settat", "Pharmacy_Type": "Urban",    "Avg_Daily_Customers": 120},
    {"Pharmacy_ID": "PH002", "Pharmacy_Name": "Pharmacie Ibn Sina",    "City": "Rabat",      "Region": "Rabat-Salé-Kénitra",      "Pharmacy_Type": "Urban",    "Avg_Daily_Customers": 95},
    {"Pharmacy_ID": "PH003", "Pharmacy_Name": "Pharmacie Shifa",       "City": "Fès",        "Region": "Fès-Meknès",              "Pharmacy_Type": "Urban",    "Avg_Daily_Customers": 85},
    {"Pharmacy_ID": "PH004", "Pharmacy_Name": "Pharmacie Al Nour",     "City": "Marrakech",  "Region": "Marrakech-Safi",          "Pharmacy_Type": "Urban",    "Avg_Daily_Customers": 110},
    {"Pharmacy_ID": "PH005", "Pharmacy_Name": "Pharmacie Salam",       "City": "Tanger",     "Region": "Tanger-Tétouan-Al Hoceïma","Pharmacy_Type": "Urban",   "Avg_Daily_Customers": 75},
    {"Pharmacy_ID": "PH006", "Pharmacy_Name": "Pharmacie Atlas",       "City": "Beni Mellal","Region": "Béni Mellal-Khénifra",   "Pharmacy_Type": "Semi-Urban","Avg_Daily_Customers": 50},
    {"Pharmacy_ID": "PH007", "Pharmacy_Name": "Pharmacie El Wifaq",    "City": "Agadir",     "Region": "Souss-Massa",             "Pharmacy_Type": "Urban",    "Avg_Daily_Customers": 80},
    {"Pharmacy_ID": "PH008", "Pharmacy_Name": "Pharmacie Erraha",      "City": "Oujda",      "Region": "Oriental",                "Pharmacy_Type": "Urban",    "Avg_Daily_Customers": 60},
    {"Pharmacy_ID": "PH009", "Pharmacy_Name": "Pharmacie Al Baraka",   "City": "Ifrane",     "Region": "Fès-Meknès",              "Pharmacy_Type": "Rural",    "Avg_Daily_Customers": 25},
    {"Pharmacy_ID": "PH010", "Pharmacy_Name": "Dépôt Central Pharma",  "City": "Casablanca", "Region": "Grand Casablanca-Settat", "Pharmacy_Type": "Warehouse","Avg_Daily_Customers": 0},
]

# 2.  CATALOGUE DE 20 MÉDICAMENTS RÉALISTES
#     Paramètres de demande calibrés par catégorie et saison

In [11]:
DRUGS = [
    # Antibiotiques
    {"Drug_ID": "D001", "Drug_Name": "Amoxicilline 500mg",     "Drug_Category": "Antibiotique",   "ATC_Code": "J01CA04", "Base_Demand": 30, "Demand_Std": 8,  "Unit_Price": 28.5,  "Max_Cap": 500, "Reorder": 80,  "Lead_Time": 3, "Is_Critical": 1, "Shelf_Life_Days": 730},
    {"Drug_ID": "D002", "Drug_Name": "Azithromycine 250mg",    "Drug_Category": "Antibiotique",   "ATC_Code": "J01FA10", "Base_Demand": 15, "Demand_Std": 5,  "Unit_Price": 55.0,  "Max_Cap": 300, "Reorder": 50,  "Lead_Time": 3, "Is_Critical": 1, "Shelf_Life_Days": 730},
    # Antalgiques / Anti-inflammatoires
    {"Drug_ID": "D003", "Drug_Name": "Paracétamol 1g",         "Drug_Category": "Antalgique",     "ATC_Code": "N02BE01", "Base_Demand": 80, "Demand_Std": 20, "Unit_Price": 8.5,   "Max_Cap": 1500,"Reorder": 200, "Lead_Time": 2, "Is_Critical": 0, "Shelf_Life_Days": 1095},
    {"Drug_ID": "D004", "Drug_Name": "Ibuprofène 400mg",       "Drug_Category": "Anti-inflam.",   "ATC_Code": "M01AE01", "Base_Demand": 55, "Demand_Std": 15, "Unit_Price": 12.0,  "Max_Cap": 1000,"Reorder": 150, "Lead_Time": 2, "Is_Critical": 0, "Shelf_Life_Days": 1095},
    {"Drug_ID": "D005", "Drug_Name": "Tramadol 100mg",         "Drug_Category": "Antalgique",     "ATC_Code": "N02AX02", "Base_Demand": 10, "Demand_Std": 4,  "Unit_Price": 42.0,  "Max_Cap": 200, "Reorder": 30,  "Lead_Time": 4, "Is_Critical": 1, "Shelf_Life_Days": 1095},
    # Antihypertenseurs
    {"Drug_ID": "D006", "Drug_Name": "Amlodipine 5mg",         "Drug_Category": "Cardio.",        "ATC_Code": "C08CA01", "Base_Demand": 40, "Demand_Std": 8,  "Unit_Price": 35.0,  "Max_Cap": 600, "Reorder": 100, "Lead_Time": 3, "Is_Critical": 1, "Shelf_Life_Days": 1460},
    {"Drug_ID": "D007", "Drug_Name": "Ramipril 5mg",           "Drug_Category": "Cardio.",        "ATC_Code": "C09AA05", "Base_Demand": 35, "Demand_Std": 7,  "Unit_Price": 38.5,  "Max_Cap": 500, "Reorder": 80,  "Lead_Time": 3, "Is_Critical": 1, "Shelf_Life_Days": 1460},
    # Antidiabétiques
    {"Drug_ID": "D008", "Drug_Name": "Metformine 850mg",       "Drug_Category": "Antidiabétique", "ATC_Code": "A10BA02", "Base_Demand": 45, "Demand_Std": 10, "Unit_Price": 22.0,  "Max_Cap": 700, "Reorder": 120, "Lead_Time": 3, "Is_Critical": 1, "Shelf_Life_Days": 1095},
    {"Drug_ID": "D009", "Drug_Name": "Insuline Glargine 100UI","Drug_Category": "Antidiabétique", "ATC_Code": "A10AE04", "Base_Demand": 12, "Demand_Std": 4,  "Unit_Price": 185.0, "Max_Cap": 150, "Reorder": 25,  "Lead_Time": 5, "Is_Critical": 1, "Shelf_Life_Days": 730},
    # Antihistaminiques
    {"Drug_ID": "D010", "Drug_Name": "Cétirizine 10mg",        "Drug_Category": "Antihistaminique","ATC_Code": "R06AE07","Base_Demand": 20, "Demand_Std": 12, "Unit_Price": 15.0,  "Max_Cap": 400, "Reorder": 60,  "Lead_Time": 2, "Is_Critical": 0, "Shelf_Life_Days": 1095},
    # Vitamines / Suppléments
    {"Drug_ID": "D011", "Drug_Name": "Vitamine D3 1000UI",     "Drug_Category": "Supplément",     "ATC_Code": "A11CC05", "Base_Demand": 50, "Demand_Std": 18, "Unit_Price": 45.0,  "Max_Cap": 800, "Reorder": 130, "Lead_Time": 2, "Is_Critical": 0, "Shelf_Life_Days": 730},
    {"Drug_ID": "D012", "Drug_Name": "Vitamine C 500mg",       "Drug_Category": "Supplément",     "ATC_Code": "A11GA01", "Base_Demand": 60, "Demand_Std": 25, "Unit_Price": 18.0,  "Max_Cap": 1000,"Reorder": 150, "Lead_Time": 2, "Is_Critical": 0, "Shelf_Life_Days": 730},
    # Antidépresseurs / Anxiolytiques
    {"Drug_ID": "D013", "Drug_Name": "Sertraline 50mg",        "Drug_Category": "Antidépresseur", "ATC_Code": "N06AB06", "Base_Demand": 8,  "Demand_Std": 3,  "Unit_Price": 75.0,  "Max_Cap": 150, "Reorder": 25,  "Lead_Time": 4, "Is_Critical": 1, "Shelf_Life_Days": 1095},
    {"Drug_ID": "D014", "Drug_Name": "Alprazolam 0.25mg",      "Drug_Category": "Anxiolytique",   "ATC_Code": "N05BA12", "Base_Demand": 6,  "Demand_Std": 2,  "Unit_Price": 35.0,  "Max_Cap": 100, "Reorder": 20,  "Lead_Time": 4, "Is_Critical": 1, "Shelf_Life_Days": 1095},
    # Gastro-entérologie
    {"Drug_ID": "D015", "Drug_Name": "Oméprazole 20mg",        "Drug_Category": "Gastro.",        "ATC_Code": "A02BC01", "Base_Demand": 35, "Demand_Std": 10, "Unit_Price": 32.0,  "Max_Cap": 600, "Reorder": 100, "Lead_Time": 2, "Is_Critical": 0, "Shelf_Life_Days": 1095},
    {"Drug_ID": "D016", "Drug_Name": "Métoclopramide 10mg",    "Drug_Category": "Gastro.",        "ATC_Code": "A03FA01", "Base_Demand": 18, "Demand_Std": 6,  "Unit_Price": 14.0,  "Max_Cap": 350, "Reorder": 60,  "Lead_Time": 2, "Is_Critical": 0, "Shelf_Life_Days": 1095},
    # Dermatologie
    {"Drug_ID": "D017", "Drug_Name": "Bétaméthasone Crème",    "Drug_Category": "Dermatologie",   "ATC_Code": "D07AC01", "Base_Demand": 12, "Demand_Std": 4,  "Unit_Price": 28.0,  "Max_Cap": 200, "Reorder": 40,  "Lead_Time": 3, "Is_Critical": 0, "Shelf_Life_Days": 1095},
    # Urgences critiques
    {"Drug_ID": "D018", "Drug_Name": "Adrénaline 1mg/ml",      "Drug_Category": "Urgence",        "ATC_Code": "C01CA24", "Base_Demand": 2,  "Demand_Std": 1,  "Unit_Price": 95.0,  "Max_Cap": 50,  "Reorder": 10,  "Lead_Time": 1, "Is_Critical": 1, "Shelf_Life_Days": 365},
    {"Drug_ID": "D019", "Drug_Name": "Morphine 10mg/ml",       "Drug_Category": "Urgence",        "ATC_Code": "N02AA01", "Base_Demand": 3,  "Demand_Std": 1,  "Unit_Price": 120.0, "Max_Cap": 60,  "Reorder": 15,  "Lead_Time": 1, "Is_Critical": 1, "Shelf_Life_Days": 365},
    # Vaccins
    {"Drug_ID": "D020", "Drug_Name": "Vaccin Grippe",           "Drug_Category": "Vaccin",         "ATC_Code": "J07BB02", "Base_Demand": 25, "Demand_Std": 30, "Unit_Price": 120.0, "Max_Cap": 300, "Reorder": 50,  "Lead_Time": 7, "Is_Critical": 1, "Shelf_Life_Days": 180},
]


# 3.  PARAMÈTRES TEMPORELS

In [12]:
START_DATE = date(2023, 1, 1)
END_DATE   = date(2024, 12, 31)
DATE_RANGE = [START_DATE + timedelta(days=i)
              for i in range((END_DATE - START_DATE).days + 1)]

# 4.  FONCTIONS DE MODÉLISATION RÉALISTE

In [13]:
def seasonal_multiplier(d: date, drug_category: str) -> float:
    """
    Modélise la saisonnalité médicale réaliste.
    Hiver → +50% antibiotiques/antihistaminiques | Été → +40% vitamines
    """
    month = d.month
    factors = {
        "Antibiotique":     [1.5,1.4,1.2,1.0,0.8,0.7,0.7,0.8,1.0,1.2,1.4,1.5],
        "Anti-inflam.":     [1.3,1.2,1.1,1.0,0.9,0.8,0.8,0.9,1.0,1.1,1.2,1.3],
        "Antalgique":       [1.2,1.1,1.0,1.0,1.0,0.9,0.9,1.0,1.0,1.1,1.1,1.2],
        "Antihistaminique": [0.7,0.7,1.0,1.4,1.6,1.5,1.3,1.2,1.3,1.1,0.8,0.7],
        "Vaccin":           [0.5,0.5,0.5,0.5,0.5,0.5,0.7,1.0,1.5,2.0,1.8,1.0],
        "Supplément":       [0.8,0.8,0.9,1.0,1.1,1.3,1.5,1.4,1.2,1.0,0.9,0.8],
        "Urgence":          [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0],
    }
    default = [1.0] * 12
    return factors.get(drug_category, default)[month - 1]


def weekend_multiplier(d: date) -> float:
    """Vendredis (+20% Maroc) et weekends légèrement différents."""
    if d.weekday() == 4:   # Vendredi
        return 1.20
    elif d.weekday() == 6: # Dimanche
        return 0.80
    return 1.0


def ramadan_multiplier(d: date) -> float:
    """Ramadan 2023 (23 mars–21 avril) et 2024 (11 mars–9 avril) → -30% ventes."""
    ramadan_periods = [
        (date(2023, 3, 23), date(2023, 4, 21)),
        (date(2024, 3, 11), date(2024, 4,  9)),
    ]
    for start, end in ramadan_periods:
        if start <= d <= end:
            return 0.70
    return 1.0


def pharmacy_size_factor(pharmacy_type: str) -> float:
    return {"Urban": 1.0, "Semi-Urban": 0.55, "Rural": 0.25, "Warehouse": 3.0}.get(pharmacy_type, 1.0)


def generate_batch_id(drug_id: str, stock_date: date) -> str:
    raw = f"{drug_id}_{stock_date.strftime('%Y%m')}"
    return "B" + hashlib.md5(raw.encode()).hexdigest()[:7].upper()


def simulate_stock_trajectory(
    pharmacy: dict, drug: dict, dates: list
) -> list[dict]:
    """
    Simule une trajectoire réaliste de stock pour un couple (pharmacie, médicament)
    sur toute la période. Intègre : réapprovisionnements, ruptures occasionnelles,
    péremptions par batch, transferts de secours.
    """
    rows = []
    size_f      = pharmacy_size_factor(pharmacy["Pharmacy_Type"])
    max_cap     = int(drug["Max_Cap"]   * size_f)
    reorder_pt  = int(drug["Reorder"]  * size_f)
    base_demand = drug["Base_Demand"]  * size_f

    # Stock initial aléatoire entre 50% et 90% capacité max
    current_stock = int(np.random.uniform(0.5, 0.9) * max_cap)

    # Batch initial : date péremption entre 60% et 100% de shelf life restante
    batch_id          = generate_batch_id(drug["Drug_ID"], dates[0])
    expiry_days_start = int(np.random.uniform(0.6, 1.0) * drug["Shelf_Life_Days"])
    expiry_date       = dates[0] + timedelta(days=expiry_days_start)
    lead_time         = drug["Lead_Time"]
    restock_pending   = 0  # countdown avant livraison

    prev_units_sold = base_demand  # pour calcul Demand_Trend

    for d in dates:
        # ── Saisonnalité & facteurs contextuels ──
        sea   = seasonal_multiplier(d, drug["Drug_Category"])
        wk    = weekend_multiplier(d)
        ram   = ramadan_multiplier(d)
        noise = np.random.normal(1.0, 0.12)  # bruit gaussien ±12%

        raw_demand = base_demand * sea * wk * ram * noise
        demand     = max(0, int(round(raw_demand)))

        # Injection d'événements rares : épidémie (+200% demande)
        if drug["Drug_Category"] == "Antibiotique" and random.random() < 0.003:
            demand = int(demand * 3.5)

        # Unités vendues bornées par le stock disponible
        units_sold    = min(demand, current_stock)
        current_stock = max(0, current_stock - units_sold)

        # Péremption du batch en cours
        days_to_expiry = (expiry_date - d).days

        # Si stock proche péremption (<= 30 j) ET abondant → gaspillage potentiel
        near_expiry_flag = int(0 < days_to_expiry <= 30 and current_stock > reorder_pt // 2)

        # Réapprovisionnement : déclenché si stock <= reorder_point OU péremption imminente
        if restock_pending > 0:
            restock_pending -= 1
            if restock_pending == 0:
                restock_qty = max_cap - current_stock
                current_stock = min(max_cap, current_stock + restock_qty)
                # Nouveau batch
                batch_id       = generate_batch_id(drug["Drug_ID"], d)
                new_expiry_offset = int(np.random.uniform(0.75, 1.0) * drug["Shelf_Life_Days"])
                expiry_date    = d + timedelta(days=new_expiry_offset)

        elif current_stock <= reorder_pt:
            restock_pending = lead_time  # commande passée → livraison dans lead_time jours

        # Flags IA pour les agents
        stockout_flag   = int(current_stock == 0)
        overstock_flag  = int(current_stock > max_cap * 0.90)
        transfer_needed = int(
            (stockout_flag == 1 or current_stock < reorder_pt * 0.5)
            and drug["Is_Critical"] == 1
        )

        # Tendance de demande (utile Agent de Prédiction)
        demand_trend = round((units_sold - prev_units_sold) / max(prev_units_sold, 1), 4)
        prev_units_sold = max(units_sold, 1)

        rows.append({
            "Date":                  d.isoformat(),
            "Pharmacy_ID":           pharmacy["Pharmacy_ID"],
            "Pharmacy_Name":         pharmacy["Pharmacy_Name"],
            "City":                  pharmacy["City"],
            "Region":                pharmacy["Region"],
            "Pharmacy_Type":         pharmacy["Pharmacy_Type"],
            "Drug_ID":               drug["Drug_ID"],
            "Drug_Name":             drug["Drug_Name"],
            "Drug_Category":         drug["Drug_Category"],
            "ATC_Code":              drug["ATC_Code"],
            "Units_Sold":            units_sold,
            "Demand_Raw":            demand,
            "Stock_Level":           current_stock,
            "Reorder_Point":         reorder_pt,
            "Max_Capacity":          max_cap,
            "Expiry_Days_Remaining": max(0, days_to_expiry),
            "Expiry_Date":           expiry_date.isoformat(),
            "Batch_ID":              batch_id,
            "Unit_Price_MAD":        drug["Unit_Price"],
            "Lead_Time_Days":        lead_time,
            "Is_Critical_Drug":      drug["Is_Critical"],
            "Stockout_Flag":         stockout_flag,
            "Near_Expiry_Flag":      near_expiry_flag,
            "Overstock_Flag":        overstock_flag,
            "Transfer_Needed_Flag":  transfer_needed,
            "Demand_Trend":          demand_trend,
        })

    return rows

# 5.  GÉNÉRATION PRINCIPALE

In [14]:
def main():
    print("=" * 70)
    print("  MAS Pharma — Génération du Synthetic Dataset")
    print(f"  Période   : {START_DATE} → {END_DATE}  ({len(DATE_RANGE)} jours)")
    print(f"  Pharmacies: {len(PHARMACIES)}  |  Médicaments: {len(DRUGS)}")
    print(f"  Lignes estimées : ~{len(DATE_RANGE)*len(PHARMACIES)*len(DRUGS):,}")
    print("=" * 70)

    all_rows = []
    total_combos = len(PHARMACIES) * len(DRUGS)
    done = 0

    for pharmacy in PHARMACIES:
        for drug in DRUGS:
            rows = simulate_stock_trajectory(pharmacy, drug, DATE_RANGE)
            all_rows.extend(rows)
            done += 1
            if done % 10 == 0 or done == total_combos:
                pct = done / total_combos * 100
                print(f"  [{done:>3}/{total_combos}]  {pct:5.1f}%  →  {pharmacy['Pharmacy_ID']} × {drug['Drug_ID']}")

    # ── Construction du DataFrame ──────────────────────────────────────────
    df = pd.DataFrame(all_rows)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Date", "Pharmacy_ID", "Drug_ID"]).reset_index(drop=True)

    # ── Statistiques de qualité ────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("  APERÇU DU DATASET GÉNÉRÉ")
    print("=" * 70)
    print(df.dtypes.to_string())
    print(f"\n  Shape           : {df.shape}")
    print(f"  Mémoire         : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    print(f"\n  Ruptures (0 stock): {df['Stockout_Flag'].sum():,} lignes  "
          f"({df['Stockout_Flag'].mean()*100:.1f}%)")
    print(f"  Péremptions proches: {df['Near_Expiry_Flag'].sum():,} lignes  "
          f"({df['Near_Expiry_Flag'].mean()*100:.1f}%)")
    print(f"  Transferts urgents : {df['Transfer_Needed_Flag'].sum():,} lignes  "
          f"({df['Transfer_Needed_Flag'].mean()*100:.1f}%)")
    print(f"  Overstocks         : {df['Overstock_Flag'].sum():,} lignes  "
          f"({df['Overstock_Flag'].mean()*100:.1f}%)")
    print(f"\n  Ventes totales (unités) : {df['Units_Sold'].sum():,}")
    top_drug = df.groupby("Drug_Name")["Units_Sold"].sum().idxmax()
    print(f"  Médicament le + vendu   : {top_drug}")

    # ── Export CSV ─────────────────────────────────────────────────────────
    output_path = "pharma_mas_dataset.csv"
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"\n   Dataset sauvegardé  →  '{output_path}'")
    print(f"      {len(df):,} lignes  ×  {len(df.columns)} colonnes")

    # ── Aperçu 5 premières lignes ──────────────────────────────────────────
    print("\n  ÉCHANTILLON (5 lignes) :")
    print(df.head().to_string(index=False))
    print("=" * 70)

    return df

# 6.  FONCTIONS UTILITAIRES POUR data_pipeline.ipynb

In [15]:
def load_dataset(path: str = "pharma_mas_dataset.csv") -> pd.DataFrame:
    """Charge le dataset avec les bons types."""
    df = pd.read_csv(path, parse_dates=["Date", "Expiry_Date"])
    return df


def get_pharmacy_network_graph(df: pd.DataFrame) -> dict:
    """
    Retourne un dictionnaire d'adjacence pour le graphe de transfert entre pharmacies.
    Utilisé par l'Agent de Décision pour calculer les routes de secours optimales.
    """
    pharmacies = df["Pharmacy_ID"].unique().tolist()
    # Matrice distance simplifiée (à enrichir avec coordonnées GPS réelles)
    distances = {
        ("PH001","PH002"): 87,  ("PH001","PH003"): 300, ("PH001","PH004"): 560,
        ("PH001","PH010"): 5,   ("PH002","PH003"): 230, ("PH002","PH005"): 346,
        ("PH003","PH009"): 70,  ("PH004","PH007"): 244, ("PH005","PH008"): 590,
        ("PH006","PH003"): 150, ("PH007","PH010"): 570, ("PH008","PH003"): 380,
        ("PH009","PH006"): 110,
    }
    # Graphe symétrique
    graph = {p: {} for p in pharmacies}
    for (a, b), dist in distances.items():
        graph[a][b] = dist
        graph[b][a] = dist
    return graph


def get_critical_alerts(df: pd.DataFrame, snapshot_date: str = None) -> pd.DataFrame:
    """
    Retourne toutes les alertes critiques pour une date donnée.
    Utilisé comme point d'entrée de l'Agent de Décision.
    """
    if snapshot_date:
        df = df[df["Date"] == snapshot_date]
    alerts = df[
        (df["Stockout_Flag"] == 1) |
        (df["Near_Expiry_Flag"] == 1) |
        (df["Transfer_Needed_Flag"] == 1)
    ].copy()
    alerts["Alert_Priority"] = alerts.apply(
        lambda r: "CRITIQUE" if r["Stockout_Flag"] and r["Is_Critical_Drug"]
                  else "HAUTE" if r["Transfer_Needed_Flag"]
                  else "MOYENNE", axis=1
    )
    return alerts.sort_values("Alert_Priority")


def get_prediction_features(df: pd.DataFrame, lag_days: int = 7) -> pd.DataFrame:
    """
    Génère les features temporelles (lags + rolling) pour l'Agent de Prédiction.
    Retourne un DataFrame prêt pour un modèle ML (LSTM, XGBoost, etc.)
    """
    df = df.sort_values(["Pharmacy_ID", "Drug_ID", "Date"])
    grp = df.groupby(["Pharmacy_ID", "Drug_ID"])

    for lag in range(1, lag_days + 1):
        df[f"Lag_{lag}d_Units_Sold"] = grp["Units_Sold"].shift(lag)

    df["Rolling_7d_Mean"]  = grp["Units_Sold"].transform(lambda x: x.rolling(7,  min_periods=1).mean())
    df["Rolling_30d_Mean"] = grp["Units_Sold"].transform(lambda x: x.rolling(30, min_periods=1).mean())
    df["Rolling_7d_Std"]   = grp["Units_Sold"].transform(lambda x: x.rolling(7,  min_periods=1).std().fillna(0))
    df["DayOfWeek"]  = df["Date"].dt.dayofweek
    df["Month"]      = df["Date"].dt.month
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

    return df.dropna(subset=[f"Lag_{lag_days}d_Units_Sold"])


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = main()


  MAS Pharma — Génération du Synthetic Dataset
  Période   : 2023-01-01 → 2024-12-31  (731 jours)
  Pharmacies: 10  |  Médicaments: 20
  Lignes estimées : ~146,200
  [ 10/200]    5.0%  →  PH001 × D010
  [ 20/200]   10.0%  →  PH001 × D020
  [ 30/200]   15.0%  →  PH002 × D010
  [ 40/200]   20.0%  →  PH002 × D020
  [ 50/200]   25.0%  →  PH003 × D010
  [ 60/200]   30.0%  →  PH003 × D020
  [ 70/200]   35.0%  →  PH004 × D010
  [ 80/200]   40.0%  →  PH004 × D020
  [ 90/200]   45.0%  →  PH005 × D010
  [100/200]   50.0%  →  PH005 × D020
  [110/200]   55.0%  →  PH006 × D010
  [120/200]   60.0%  →  PH006 × D020
  [130/200]   65.0%  →  PH007 × D010
  [140/200]   70.0%  →  PH007 × D020
  [150/200]   75.0%  →  PH008 × D010
  [160/200]   80.0%  →  PH008 × D020
  [170/200]   85.0%  →  PH009 × D010
  [180/200]   90.0%  →  PH009 × D020
  [190/200]   95.0%  →  PH010 × D010
  [200/200]  100.0%  →  PH010 × D020

  APERÇU DU DATASET GÉNÉRÉ
Date                     datetime64[ns]
Pharmacy_ID                 